In [ ]:
import pandas as pd
import json
import re

base_path = ""

def clean_and_parse_result(raw):
    try:
        cleaned = raw.strip('"').replace('\\"', '"')
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {}

def load_and_parse(files, atc_code):
    dfs = []
    for f in files:
        df = pd.read_csv(base_path + f)

        # Parse 'result' JSON
        parsed_results = df["result"].apply(clean_and_parse_result)
        result_df = pd.json_normalize(parsed_results)

        # Drop old 'result' and add parsed columns + ATC code tag
        df = df.drop(columns=["result"]).reset_index(drop=True)
        df = pd.concat([df, result_df], axis=1)
        df["atc_code"] = atc_code

        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Load and tag each group
a10_df = load_and_parse(["a10_switch_stops.csv"], "a10")

a10_original = pd.read_csv("a10_switches_original.csv").dropna(subset=['anamnesis'])

def combine_dfs(original, df):
    combined = pd.concat([original.reset_index(drop=True), df.drop(columns=['id']).reset_index(drop=True)], axis=1)
    return combined

# Combine each pair
combined_a10 = combine_dfs(a10_original, a10_df)

# Concatenate all together
final_combined_df = combined_a10

# Optional: Check the result
print(final_combined_df.head())

unique_anamnesis_df = final_combined_df.drop_duplicates(subset='anamnesis')

reason_df = unique_anamnesis_df[unique_anamnesis_df['reason_for_stopping'] != ""]
print(len(reason_df))

In [ ]:
diabetes_drugs = [
    "metformin",
    "metformiin",
    "metformiin",
    "metformin",
    "metformiini",
    "metformini",
    "metformini",
    "metformin",
    "metforali",
    "metforal",
    "victoza",
    "diaprel",
    "jardiance",
    "amaryl",
    "janumet",
    "trajenta",
    "forxiga",
    "toujeo",
    "lantus",
    "levemir",
    "januvia",
    "insuliin",
    "insuliini",
    "insuliinpump",
    "xigduo",
    "jentadueto",
    "gliklada",
    "victozat",
    "diapreel",
    "jardiance",
    "victoza",
    "metforminum",
    "sglt-2",
    "insuliinravi",
    "gliklada",
    "victoza",
    "amaryl",
    "metformiin ja jardiance",
    "t.diaprel",
    "diabeedi",
    "gliclada",
    "humalog",
    "novomix",
    "novorapid",
    "tresiba",
    "nrapid",
    "metformi",
    "biguanidi",
    "glimeperid",
    "gliclada",
    "glicladat",
    "gliclazidi",
    "januuvia",
    "saksagliptiin",
    "jardians",
    "synjardi",
    "jardiace",
    "pioglitazon",
    "comboglyze2.5/1.0"
]


# Make sure all terms are lowercase
diabetes_drugs = [drug.lower() for drug in diabetes_drugs]

# Join the list into a single regex pattern (escaped, joined with | for "OR" logic)
pattern = "|".join(re.escape(drug) for drug in diabetes_drugs)

# Filter drug names that contain any diabetes-related term (case-insensitive)
diabetes_df = reason_df[reason_df['drug_name'].str.lower().str.contains(pattern, na=False)]



In [ ]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from transformers import AutoTokenizer
import torch
from openai import OpenAI

import json
import csv
import re
import math
import numpy as np
import pandas as pd
import asyncio
import matplotlib.pyplot as plt 

from pydantic import BaseModel
from enum import Enum

from huggingface_hub import login
HF_TOKEN = ""
login(HF_TOKEN)  # This logs you in for the session

import requests
response = requests.get("https://huggingface.co")
print(response.status_code)  # Should print 200 if the connection is successful

model_name = "neuralmagic/Meta-Llama-3.1-70B-Instruct-quantized.w4a16"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
! nvidia-smi

llm = LLM(model=model_name, device=device, max_model_len=65536, tensor_parallel_size=1, enable_prefix_caching=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Define your classification categories

categories = [
    "Doctor", # 
    "Patient",
    "Unspecified"#
]

example_reasons = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    ""
    
]

example_anamneses = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    ""
    
]
example_labels = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    ""
]

# Set up guided decoding to restrict outputs to your categories
guided_decoding = GuidedDecodingParams(choice=categories)

# Configure sampling parameters with guided decoding
sampling_params = SamplingParams(guided_decoding=guided_decoding)


system_prompt = {
"role": "system", "content":"""You are a helpful assistant that reads Estonian electronic health records written by doctors diabetes medication discontinuation for their patients with high blood sugar levels. For each input, determine who made the decision to stop the use of a medication: the patient or the doctor.
Your input is the extracted phrase about stopping the drug and the full anamnesis for context.

Here are the categories you must use:

1. Doctor: The doctor made the decision that the patient should stop the medications.
2. Patient: The patient stopped taking the medications without a clear signal from the doctor.
3. Unspecified: Based on the text, it is unclear who made the decision.

Only choose one category per input. Respond only with the category name."""}


messages_template = [system_prompt]

drug_stop_list = diabetes_df['drug_stop_phrase'].tolist()
anamnesis_list = diabetes_df['anamnesis'].tolist()

# reason_list = reason_list[:10]
# anamnesis_list = anamnesis_list[:10]

for reason, anamnesis, label in zip(example_reasons, example_anamneses, example_labels):
    user_content = f"Discontinuation phrase:\n{reason}\nAnamnesis:\n{anamnesis}"
    messages_template.append({"role": "user", "content": user_content})
    messages_template.append({"role": "assistant", "content": label})


messages_list = [
    messages_template + [{"role": "user", "content": "Discontinuation phrase:\n" + str(reason) + "\nAnamnesis:\n" + str(anamnesis)}]
    for reason, anamnesis in zip(drug_stop_list, anamnesis_list)
]

prompts = [
    tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    for messages in messages_list
]

# Create the prompt for classification

# Generate the classification
outputs = llm.generate(
    prompts=prompts,
    sampling_params=sampling_params
)

# Extract and print the classification result
# classification = outputs[0].outputs[0].text.strip()

In [ ]:
classified_categories = [output.outputs[0].text.strip() for output in outputs]

# Build a dataframe
df = pd.DataFrame({
    "Input": drug_stop_list,
    "Category": classified_categories
})

# Save to CSV
df.to_csv("a10_switches_who_stopped.csv", index=False)

# Optional: print path or success message
print("Results saved to a10_switches_who_stopped.csv")